# USB-6289 driver — how to use it

A tutorial for the QCoDeS driver in `NIUSB_6289_v2`. It goes from the simplest
thing (set a voltage, read it back) to the hardest (record for hours against an
atomic clock and watch it live), and every function is explained where it is
first used.

**Hardware assumed** — a loopback and the rubidium 1 pps:

| signal | terminal | ground |
|---|---|---|
| AO 0 → AI 0 | 15 → 1 | AO GND 16 → AI GND 3 |
| 1 pps → PFI 9 | 83 | D GND 82 |

**Restart the kernel** whenever you edit the driver. Python caches imported
modules, so an edited file has no effect until you do — this has caused more
confusion than any actual bug.

## 0. Imports

The driver is split into files by job:

| module | what it holds | needs hardware? |
|---|---|---|
| `usb6289` | `USB6289`, the instrument you construct | yes |
| `postprocess` | reading and analysing recorded runs | **no** — numpy only |
| `livescope` | viewing a run while it is being written | **no** — numpy only |

`postprocess` and `livescope` import nothing but numpy, so you can analyse a
week-long recording on a laptop that has never had NI-DAQmx installed.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from NIUSB_6289_v2.usb6289 import USB6289
from NIUSB_6289_v2.postprocess import (load_long_run, load_segment, load_run,
                                       iter_segments, channels_of, edge_report,
                                       clean_edges, times_from_edges)
from NIUSB_6289_v2.livescope import LiveScope, RunFiles

## 1. Connect

```python
USB6289(name, device="Dev1")
```

* `name` — the QCoDeS instrument name. Must be unique in the session; making a
  second `USB6289("daq")` raises. If you re-run this cell, call `daq.close()`
  first.
* `device` — the DAQmx device name, as shown in NI MAX.

On construction the driver **asks the board what it is** — how many channels,
which counters, the maximum rates, the available input ranges — and builds its
validators from the answer. It does not trust a data sheet.

In [ ]:
daq = USB6289("daq", device="Dev1")

### The instrument tree

Everything hangs off `daq` in three subsystems:

```
daq                     the board
├── daq.ai              analog INPUT task    (one clock, one trigger)
│   ├── daq.ai.ai0 … ai31        32 input pins
│   └── daq.ai.channels          the same 32, as a group
├── daq.pps             the 1 pps counter    (no channels: one counter, one job)
└── daq.ao              analog OUTPUT task   (one clock, one waveform)
    ├── daq.ao.ao0 … ao3         4 output pins
    └── daq.ao.channels          the same 4, as a group
```

The rule for *where a setting lives*: *who does the hardware let decide it?*

* decided **per pin** → on the channel: `ai0.v_max`, `ao0.dc`
* decided **once per task** → on the subsystem: `daq.ai.rate`, `daq.ao.freq`
  (there is one ADC, so you cannot give two inputs different sample rates)
* needs **two subsystems** → on the instrument: `daq.clock_check`, `daq.long_run`

In [ ]:
daq.ai.channels.enabled()          # broadcast: read a parameter on all 32 at once

---
# 1. Loopback test

AO 0 wired to AI 0: the board generates a signal and measures its own output.
Everything downstream depends on this working.

## 2.1 Constant voltage

### `daq.ai.setup(...)` — configure the input in one call

```python
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=0.05)
```

* **first argument** says which pins are on **and** their range. It accepts
  * `"ai0"` — enable it, leave the range alone
  * `["ai0", "ai3"]` — several
  * `{"ai0": 2.0}` — a number means **± that**
  * `{"ai0": (-0.5, 1.5)}` — a pair means exactly that
  * each input has its own programmable gain, so mixing ranges is free
* `rate` — samples per second **per channel**
* `duration` — seconds per `acquire()`
* `trigger` — a terminal to wait for; `None` starts immediately
* **enabling is exclusive**: pins you do not name are switched off
* **returns the channel objects**, so `ai0, =` gives you a handle in one line

### `daq.ao.setup(...)` — configure the output

```python
ao0, = daq.ao.setup(channels_role={"ao0": {"dc": 0.0}})
```

Each pin has a **role**, because the board can drive it two different ways:

| role | meaning |
|---|---|
| `"off"` | not in the task |
| `"dc"` | held at a constant voltage by an untimed write |
| `"wave"` | generating the waveform — **at most one pin**, the board has one AO clock |

Going to `"off"` does *not* change the pin's voltage: an untimed output holds
its last value indefinitely. Set `dc(0.0)` first if you want it at zero.

In [ ]:
# configuration for AnalogInput
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=0.05)

# configuration for AnalogOutput: ao0 as a DC level, starting at 0 V
ao0, = daq.ao.setup(channels_role={"ao0": {"dc": 0.0}})

### `daq.check()` — validate before touching the hardware

```python
daq.check(verbose=True, strict=False)
```

Returns a list of `(level, where, message)`, empty if all is well.
`strict=True` raises on any error instead of returning.

It catches what no single validator can, because the constraint spans
parameters or lives in the driver:

* `amp + |offset|` against the ±10 V output range
* `rate × channels` against the board's **aggregate** limit
* a DIFF channel whose negative half is separately enabled
* a waveform buffer too big for the onboard FIFO
* a record too large to hold in RAM
* **a trigger terminal that cannot be routed** — it hands the task to DAQmx for
  verification, which programs the clocks without starting them

That last one is the prize: it is the failure that otherwise appears only when
the run starts, possibly hours later.

In [ ]:
daq.check()

### `daq.wiring()` — which screw terminals to connect

```python
rows = daq.wiring(verbose=True)
```

Prints a table **and returns it** as a list of dicts
(`role, channel, connection, terminal, signal, note`), so it goes straight into
a DataFrame or a JSON file recording what was physically plugged in.

It reads the **live** configuration, so it changes as you enable channels or
move the waveform. Every signal row is followed by its **return** row: on a
multiplexed board a floating input does not read zero, it reads whatever the
multiplexer left behind.

In [ ]:
daq.wiring()

### Where do I plug this in?

Two different questions, two tools.

**The static question** — "which terminal is `ai3`?" The answer never changes;
it is the connector.

| call | returns |
|---|---|
| `daq.pin_of("ai3")` | `10` — accepts `ai3`, `AI 3`, `Dev1/ai3`, `PFI8`, `AI GND` |
| `daq.signal_at(81)` | `'PFI 8/P2.0'` — the inverse: what *is* this terminal? |
| `daq.nearest_ground(15, "AO GND")` | `16` — the ground pin to actually wire to |
| `daq.pinout()` | the whole 128-pin connector, both halves |

`nearest_ground` takes the ground **family** — `"AI GND"`, `"AO GND"`,
`"D GND"` — and which one you use matters: returning a signal through the
wrong ground is how you get a mains hum you cannot explain.

**The live question** — "what do I plug in for what I have configured right
now?" That is `daq.wiring()`, further down. It changes as you enable channels
or move the waveform, and it pairs every signal with its return.

In [ ]:
print("ai3 is on terminal ", daq.pin_of("ai3"))
print("its AI GND is       ", daq.nearest_ground(daq.pin_of("ai3"), "AI GND"))
print("terminal 81 carries ", daq.signal_at(81))

daq.pinout()          # the full connector

### Which pins can be a trigger, and which can carry the 1 pps?

```python
daq.terminals(physical=True)     # only the ones a cable can reach
daq.terminals(physical=False)    # everything DAQmx will route
```

The board reports **two kinds** of routable terminal, and the difference
decides what each can be used for:

| kind | example | can an external cable reach it? |
|---|---|---|
| **physical** | `/Dev1/PFI8` → terminal 81 | **yes** |
| **internal** | `/Dev1/ai/SampleClock`, `/Dev1/Ctr0InternalOutput` | no — no pin exists |

* `ai.trigger` and `ao.trigger` accept **either**. An internal route is how one
  subsystem triggers another without a wire.
* `pps.terminal` must be **physical** — the rubidium's pulse arrives from
  outside the board, so it needs a pin.

Both parameters validate against this list, so a terminal that cannot be
routed is rejected at set time; and `daq.check()` goes further by handing the
task to DAQmx, which rejects routes that are legal in principle but conflict
with what else is configured.

### What if the trigger and the 1 pps are on *different* pins?

**Nothing breaks — that is the ordinary case.** They are independent settings:

* `ai.trigger` decides *when the acquisition starts*
* `pps.terminal` decides *where the atomic seconds arrive*

Putting both on PFI 8 is a convenience, not a requirement. It buys one thing:
the same edge starts the acquisition and marks an atomic second, so **scan 0
is an atomic second** and the first edge comes back at scan 0.

On separate pins, the first edge lands wherever it lands — somewhere in the
first second — and `times_from_edges` still puts `t = 0` at that first edge.
Samples taken *before* it come out **negative**, which is correct and
deliberate: they happened before the first atomic second you witnessed.

That is often what you want. With `ai.trigger` on the OPX marker and
`pps.terminal` on PFI 9, the first edge tells you exactly how far the next
atomic second was from the start of the shot — which is the number that puts a
QM experiment on the atomic timebase.

**One cost of sharing the pin**, worth knowing: the acquisition then waits for
the next 1 pps edge, so it starts up to 1 s late — and if the rubidium is off
or unplugged, it waits for the 60 s read timeout instead of starting at all.

In [ ]:
daq.terminals()                    # physical: a cable can reach these

# ai.trigger and ao.trigger can also use internal routes:
# daq.terminals(physical=False)

### So where can the 1 pps actually go?

**Any PFI line — PFI 0 to PFI 15.** All sixteen are on the digital connector
and any of them can clock a counter, so the choice is about wiring, not
capability.

`daq.terminals()` lists them with their **ground** pin, and the board is not
even-handed:

```
   /Dev1/PFI0    terminal  73   D GND  82 (9 pins away)
   /Dev1/PFI7    terminal  80   D GND  82 (2 pins away)
   /Dev1/PFI8    terminal  81   D GND  82 (adjacent)
   /Dev1/PFI9    terminal  83   D GND  82 (adjacent)
   /Dev1/PFI15   terminal  95   D GND  94 (adjacent)
```

**PFI 8–15 each sit next to a D GND pin. PFI 0–7 do not** — they all share the
single D GND at pin 82, up to nine screws away.

For an external TTL that matters twice over: a short return loop picks up less,
and two adjacent screws are simply easier to get right at the terminal block at
midnight. So for the rubidium, use **PFI 8–15**.

| line | signal pin | ground pin |
|---|---|---|
| PFI 8 | 81 | 82 |
| PFI 9 | 83 | 82 |
| PFI 10 | 85 | 84 |
| PFI 11 | 87 | 86 |
| PFI 12 | 89 | 88 |

The convention in this driver's docstrings is **PFI 8 for the trigger, PFI 9
for the 1 pps** — arbitrary, but consistent, and `daq.wiring()` prints whatever
you actually chose.

One thing that is *not* a constraint: the counter. `pps.counter` defaults to
`ctr1`, and any PFI line can clock either counter, so the two choices are
independent.

### Is any pin *dedicated* to the trigger, or to the 1 pps?

**No — and that is a property of this board, not an omission.** On an M Series
device every PFI line is fully routable to every timing signal: start trigger,
sample clock, counter source, counter gate. "Which pin is the trigger pin?" has
no answer beyond *the one you chose*.

(It was a real question on the older **E Series**, where PFI 0 *was* the AI
start trigger and nothing else. That is not this board — if you are reading an
older manual or an older script, that is the difference.)

What the manual *does* define is the pin each counter uses **when you do not
say otherwise**:

| line | default role | line | default role |
|---|---|---|---|
| PFI 3 | CTR 1 SRC | PFI 10 | CTR 0 AUX |
| PFI 4 | CTR 1 GATE | PFI 11 | CTR 1 AUX |
| PFI 8 | CTR 0 SRC | PFI 12 | CTR 0 OUT |
| PFI 9 | CTR 0 GATE | PFI 13 | CTR 1 OUT |
| | | PFI 14 | FREQ OUT |

These are **not reservations**. This driver routes the counter's sample clock
explicitly, so `pps.terminal` works on any PFI line including the ones above —
which is why PFI 9 is used here despite being CTR 0's default gate. They matter
only because a counter left on its defaults *in some other script* will quietly
take those lines.

`daq.terminal_roles()` puts it together: every PFI line, its pins, what **you**
are using it for right now, and what the board would use it for by default. It
reads the defaults back **from the device** where it can, falling back to the
table above.

In [ ]:
daq.terminal_roles()      # who is using what, and what the defaults are

### The sweep

* `ao0.dc(v)` — set the level. Applied **immediately**; no clock, nothing to
  start. Setting `dc` on a pin whose role is `"off"` promotes it to `"dc"`.
* `daq.ai.acquire()` — run **one** acquisition and return
  `{channel_name: array}`. This is the only call that touches the AI hardware.
* `time.sleep(0.1)` — let the output settle before measuring.

In [ ]:
v_min, v_max = -1.0, 1.0
step = 0.5

V = np.arange(v_min, v_max + step, step)
V

In [ ]:
for v in V:
    ao0.dc(v)                       # set the output
    time.sleep(0.1)                 # let it settle
    data = daq.ai.acquire()         # measure - returns {channel: array}
    trace = data["ai0"]

    print(f"set {v:+.2f} V  ->  read {trace.mean():+.4f} V "
          f"(+/- {trace.std():.4f}, {trace.size} samples)")

ao0.dc(0)                           # leave the output at zero

`daq.ao.channels.dc()` is a **broadcast read** over the `ChannelList`: it
returns the parameter for all four pins at once, rather than asking each in
turn. The same works for any parameter the channels share.

In [ ]:
daq.ao.channels.dc()

## 2.2 Waveform

### Waveform settings live on `daq.ao`, not on the pin

The board has **one** AO clock, so shape, frequency, amplitude and update rate
belong to the subsystem. Only *which pin carries it* is per-pin.

| parameter | meaning |
|---|---|
| `daq.ao.shape` | `"square"`, `"sine"`, `"triangle"`, `"ramp"` |
| `daq.ao.freq` | Hz — snapped so a whole number of samples fits one period |
| `daq.ao.amp` | volts, peak |
| `daq.ao.offset` | volts |
| `daq.ao.rate` | DAC update rate, S/s — sets edge resolution and buffer length |
| `daq.ao.duty` | square duty cycle |
| `daq.ao.actual_freq` | what you really got after snapping |
| `daq.ao.onboard` | **True if the waveform loops from the board's own memory** |

`onboard` matters for long runs: if the buffer fits the onboard FIFO the loop
needs no USB traffic and a saturated input stream cannot starve it. If it does
not fit, the waveform is pushed across USB continuously and will eventually
underflow (`-200621`).

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100)
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=5)

print("asked for  ", daq.ao.freq(), "Hz")
print("buffer     ", f"{daq.ao.buffer_len():,} samples of {daq.ao.fifo_samples():,} FIFO")
print("onboard?   ", daq.ao.onboard(), " <- must be True for a long run")

### `with daq.ao.generating():` — generate for the duration of a block

```python
with daq.ao.generating(**overrides):
    data = daq.ai.acquire()
```

Equivalent to `daq.ao.start()` … `daq.ao.stop()`, but the stop is **guaranteed
even if the measurement raises** — which matters, because a buffered output
otherwise holds its last value forever.

Any `ao` parameter can be overridden for the block and is restored afterwards:

```python
with daq.ao.generating(freq=200, amp=0.5):
    ...
```

In [ ]:
with daq.ao.generating():
    data = daq.ai.acquire()

print(f"{len(data['ai0']):,} samples, {data['ai0'].min():+.3f} .. {data['ai0'].max():+.3f} V")

### `trace` and `time_axis` — the QCoDeS view of the same data

* `ai0.trace()` — **never measures**. The hardware produces every enabled
  channel in one block, and `acquire()` split it into rows; `trace` just picks
  this channel's row. It **raises** if there is no scan in hand, rather than
  handing back a stale array.
* `daq.ai.time_axis()` — sample index ÷ *actual* rate. The clock divides a
  20 MHz timebase, so the rate the hardware programmed is rarely exactly the
  one you asked for, and only the real one gives a correct axis.

So there is exactly one line in your code where the hardware runs, and it is
one you wrote — `daq.ai.acquire()`.

In [ ]:
y = ai0.trace()                  # picks ai0's row out of the last acquisition
t = daq.ai.time_axis()           # seconds, from the ACTUAL sample rate

print(f"requested rate {daq.ai.rate():,.0f} S/s")
print(f"actual rate    {daq.ai.actual_rate():,.3f} S/s   <- use this one")

In [ ]:
start, end = 0, 1000

plt.plot(t[start:end] * 1e3, y[start:end])
plt.xlabel("t [ms]"); plt.ylabel("V [V]")
plt.grid(alpha=0.3)

## 2.3 Waveform, triggered

A **start trigger** makes a task wait for an edge on a terminal instead of
starting immediately. Point `ai.trigger` and `ao.trigger` at the same line and
both start on one edge, so the waveform has a known phase at sample 0.

Here both triggers are on **PFI 8**, while the 1 pps arrives separately on
**PFI 9**. So the acquisition starts when the marker fires, and the atomic
seconds are recorded independently — sample 0 is *not* an atomic second, and
the first edge lands somewhere in the first second.

That is the general case and the one worth learning. Putting both on the same
pin is a special case covered below.

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100,
                    trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=5,
                    trigger="/Dev1/PFI8")

daq.check()

In [ ]:
with daq.ao.generating():
    data = daq.ai.acquire()      # blocks until the PFI8 edge arrives

y = ai0.trace()
t = daq.ai.time_axis()

plt.plot(t[:1000] * 1e3, y[:1000])
plt.xlabel("t [ms]"); plt.ylabel("V [V]"); plt.grid(alpha=0.3)

---
# 2. Clock check — the board against the rubidium

The board's clock does not know it is wrong. It divides its 20 MHz timebase,
reports a sample rate with confidence, and drifts by seconds over a day.

The FS725 emits one pulse per **atomic** second. Comparing them is what makes a
multi-day time axis mean anything.

### How the counter is used, and why it looks backwards

| | |
|---|---|
| what is **counted** | `ai/SampleClock` — the board's own sample clock |
| what **latches** it | the 1 pps, used as the counter task's *sample clock* |

So each value the counter returns is *the scan index that was current when an
atomic second arrived*. That is the edge table — complete, edge-triggered,
nothing missed, in the same units as the data.

Feeding the 1 pps into an analog channel instead would miss ~75 % of the pulses
at 25 kS/s, because the ADC samples instantaneously and the pulse is 10 µs wide.

### `daq.clock_check(seconds=None, verbose=True)`

Acquires for `seconds`, then reports the true rate. If a pin has role
`"wave"`, the waveform runs during the measurement and is measured back as a
cross-check — but note it can only ever **confirm**, never detect: the DAC and
the ADC divide the same timebase, so they are wrong together.

**How long to run it** — each edge is resolved to ±1 scan (40 µs at 25 kS/s),
and the estimate spans the whole measurement:

| `seconds=` | resolution | equivalent |
|---|---|---|
| 10 | ~6 ppm | ±0.5 s/day |
| 60 | ~1 ppm | ±0.09 s/day |
| 600 | ~0.1 ppm | ±0.01 s/day |

Use 10 s to confirm the wiring. Use 600 s once, for the number you would quote.

In [ ]:
daq.pps.terminal("/Dev1/PFI9")     # the 1 pps line
daq.pps.counter("ctr1")            # leaves ctr0 free for anything else

ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100,
                    trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=5,
                    trigger="/Dev1/PFI8")
daq.check()

In [ ]:
result = daq.clock_check(seconds=10)

### Reading the output

* **edges caught** should be about `seconds`. Far fewer means the pulse is not
  reaching the terminal — check `daq.wiring()`.
* **discarded values** are normal: every acquisition ends with one latched
  value that is *not* an atomic second (when the AI task stops the sample clock
  stops, the count freezes, and one more value comes back carrying it).
* **ppm** in the tens is normal — the board is specified to ±50 ppm. What
  matters is that it is *stable*, because that is what makes it correctable.
* **the square-wave line**, if present, should agree with the rubidium.

---
# 3. The atomic time axis

### `daq.acquire_with_pps()`

Returns `(data, edges)` — one record on every enabled channel, plus the atomic
seconds that fell inside it, as **scan indices**. An event at scan *k* and an
atomic second at scan *e* are therefore on the same ruler: no clock conversion,
no assumption about the rate.

It lives on the instrument, not on a submodule, because it drives two: the
counter has to start **before** the AI clock runs, and be stopped and joined
before its queue is drained.

### Two time axes

| | |
|---|---|
| `daq.ai.time_axis()` | scan index ÷ actual rate — the **board's** clock |
| `daq.ai.atom_time_axis()` | interpolated between 1 pps edges — the **rubidium** |

The second one never uses the sample rate. It asks *where does this sample sit
between two atomic ticks?* — which is what absorbs the drift.

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100,
                    trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, duration=60 * 5,
                    trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI9")
daq.pps.counter("ctr1")
daq.check(strict=True)

In [ ]:
# generate the output signal while measuring, and collect the atomic seconds
with daq.ao.generating():
    data, edges = daq.acquire_with_pps()

y = data["ai0"]
print(f"{len(y):,} samples, {len(edges)} atomic seconds")

In [ ]:
t_board  = daq.ai.time_axis()        # the board's own clock
t_atomic = daq.ai.atom_time_axis()   # corrected against the rubidium

print(f"board  {t_board[0]:+.4f} .. {t_board[-1]:,.4f} s")
print(f"atomic {t_atomic[0]:+.4f} .. {t_atomic[-1]:,.4f} s")
print(f"difference at the end: {(t_atomic[-1] - t_board[-1]) * 1e3:+.3f} ms")

### The drift plot

`t_atomic - t_board` is the board's clock walking away from the rubidium. The
slope is the error in ppm; the **staircase** is real, not noise: each edge is
quantised to one scan (40 µs) and the correction is linear between edges, so
each step lasts one second.

The dashed lines mark the atomic seconds.

In [ ]:
start, end = 0, len(t_atomic) - 1000
fig, ax = plt.subplots(figsize=(14, 5))

for edge in edges[::10]:                 # every 10th, or the plot is solid black
    if edge > end:
        break
    ax.axvline(t_atomic[edge], color="black", alpha=0.3, lw=1, ls="--")

res = t_atomic[start:end] - t_board[start:end]
ax.plot(t_atomic[start:end], res * 1e6, color="red", lw=2)

ppm = np.polyfit(t_board[start:end], res, 1)[0] * 1e6
ax.set_xlabel("t [s]"); ax.set_ylabel(r"$\Delta t$ [$\mu$s]")
ax.set_title(f"clock error {ppm:+.2f} ppm  ->  {abs(ppm) * 0.0864:.2f} s/day")
ax.grid(alpha=0.3)

---
# 4. Long run — recording to disk

### `daq.long_run(outdir, ...)`

| argument | meaning |
|---|---|
| `outdir` | a **new** directory. It refuses to reuse one — two runs in a directory cannot be told apart afterwards |
| `hours` | how long. `None` runs until you stop it |
| `rotate_minutes` | `None` (default) = **one file per channel** for the whole run. `60` = a new file every hour |
| `background` | `True` returns a handle immediately and records in a thread |
| `require_pps` | `False` allows a run with no rubidium (bench testing) |
| `live_decimate` | how many scans per summary block for `live_preview` (100) |
| `live_seconds` | how far back that summary reaches (60) |
| `to_hdf5` | also convert closed segments to `.h5` in the background |

`live_decimate` and `live_seconds` **do not affect what is recorded** — only
the little in-RAM buffer you glance at.

### What ends up on disk

```
outdir/
    run.json        the instrument settings at the start
    manifest.jsonl  which file holds which scans — one JSON per line
    edges.i64       every 1 pps edge of the whole run, as scan indices
    seg_00000_ai0_20260813T101500Z.f32     the samples: raw float32
```

Nothing is filtered before writing. The `.f32` is what DAQmx returned; the
edge table includes its artefacts. **A recording you cannot re-take must not be
stored already interpreted** — if the interpretation turns out to be wrong, the
raw file is what lets you fix it tomorrow instead of measuring again.

In [ ]:
ao0, = daq.ao.setup({"ao0": {"wave": 1.0}}, shape="square", freq=100,
                    trigger="/Dev1/PFI8")
ai0, = daq.ai.setup({"ai0": (-2.0, 2.0)}, rate=25_000, trigger="/Dev1/PFI8")

daq.pps.terminal("/Dev1/PFI9")
daq.pps.counter("ctr1")

### Check twice before committing hours

`onboard` must be **True**. If it is False the waveform is regenerated across
USB and will underflow after a few hours of competing with the input stream —
a failure that appears in the middle of the night, not now.

In [ ]:
daq.check(strict=True)                       # raises if anything cannot run

print("AO regenerates onboard:", daq.ao.onboard(),
      f"({daq.ao.buffer_len():,} of {daq.ao.fifo_samples():,} FIFO samples)")

rate = daq.ai.actual_rate()
print(f"disk rate: {rate * 4 / 1e6:.2f} MB/s = {rate * 4 * 86400 / 1e9:.1f} GB/day")

### Rehearse for 3 seconds

Look at the signal and the edges before spending hours. A wiring mistake found
here costs three seconds.

`edge_report(edges, nominal_rate)` says whether a run's timing is sound: how
many latched values were discarded and why, how many atomic seconds were
**missed** (kept as gaps — the numbering accounts for them), and the board's
rate and drift measured across the whole record.

In [ ]:
daq.ai.duration(3.0)
data, edges = daq.acquire_with_pps()

print(f"{len(data['ai0']):,} samples, "
      f"{data['ai0'].min():+.3f} .. {data['ai0'].max():+.3f} V\n")
edge_report(edges, nominal_rate=daq.ai.actual_rate())

### Start

AO first, so it is armed before the input task starts. `background=True`
returns immediately — this cell does not block, and the acquisition continues
while you run the cells below.

**Put `OUTDIR` on a local disk.** A sync client (OneDrive, Dropbox) trying to
upload a continuously growing file will stall the write, and a stalled write
becomes a buffer overflow.

In [ ]:
daq.ao.start()                     # arms; fires on the next PFI8 edge

OUTDIR = "long_run_test"           # a NEW directory each time

run = daq.long_run(OUTDIR,
                   hours=0.1,      # 6 minutes
                   rotate_minutes=2,
                   background=True)
run

## 5.1 Watching it — the two paths

There are two places live data exists, and choosing the wrong one is the usual
confusion:

| | `daq.live_preview()` | the files (`LiveScope`, `RunFiles`) |
|---|---|---|
| where | a deque in RAM | disk |
| how much | last `live_seconds` (60) | **everything** |
| channels | the first one only | all of them |
| resolution | min/max per `live_decimate` block | every sample |
| answers | *"is it alive?"* | *"what did it measure?"* |

`run.status()` gives progress: seconds recorded, current segment, and whether
the thread is still alive.

In [ ]:
s = run.status()
print(f"{s['seconds']:,.1f} s recorded, segment {s['segment']}, alive={run.alive}")

t, y = daq.live_preview()
print(f"live buffer: {len(y):,} points spanning {t[-1] - t[0]:.1f} s"
      if len(y) else "live buffer empty")

## 5.2 Live plot

**Two things must happen every frame**, and missing either is why a live plot
appears not to work:

1. **clear the axes** — `ax.plot()` in a loop *adds* a line each time, so after
   a few hundred frames the figure is hundreds of lines and grinds to a halt;
2. **push the figure to the screen** — with the default `%matplotlib inline`
   backend there is no GUI event loop, so `plt.pause()` redraws nothing and the
   cell just blocks. The figure has to be re-`display`ed with the previous
   output cleared.

`time.sleep()` alone never works on inline: it sleeps without redrawing.

> **A note on what you are looking at.** The live buffer stores the **min and
> max** of each block of `live_decimate` samples, not every 100th sample.
> Striding would be undersampling — at 250 effective S/s a 100 Hz square has
> 2.5 samples per period and renders as ragged nonsense with the wrong duty
> cycle, while the file is perfect. Min/max cannot misrepresent the envelope,
> but the *order within a block* is not real. For the true waveform, use §5.3.

In [ ]:
# RECIPE A - works with the DEFAULT inline backend, nothing to install
from IPython.display import display, clear_output

fig, ax = plt.subplots(figsize=(12, 4))

try:
    while run.alive:
        t, y = daq.live_preview()
        ax.clear()                                  # (1) or the lines pile up
        if len(y):
            ax.plot(t, y, lw=1)
            ax.set_xlabel("board time [s]"); ax.set_ylabel("V [V]")
            ax.set_title(f"{run.status()['seconds']:,.0f} s recorded")
            ax.grid(alpha=0.3)
        clear_output(wait=True)                     # (2) wait=True = no flicker
        display(fig)
        time.sleep(0.5)
except KeyboardInterrupt:
    print("stopped plotting (the run continues)")

**Recipe B** is better if you install `ipympl` (`pip install ipympl`): put
`%matplotlib widget` in its own cell above and re-run from there. You get real
pan and zoom, no flicker, and updating one line's data is much cheaper than
redrawing.

In [ ]:
# RECIPE B - needs %matplotlib widget
fig, ax = plt.subplots(figsize=(12, 4))
line, = ax.plot([], [], lw=1)
ax.set_xlabel("board time [s]"); ax.set_ylabel("V [V]"); ax.grid(alpha=0.3)

try:
    while run.alive:
        t, y = daq.live_preview()
        if len(y):
            line.set_data(t, y)          # update the SAME line - no pile-up
            ax.relim(); ax.autoscale_view()
        plt.pause(0.5)                   # redraws AND sleeps
except KeyboardInterrupt:
    pass

## 5.3 An oscilloscope view

What makes a scope readable is not the drawing — it is the **trigger**. Without
it the waveform slides across the screen every frame; with it, it stands still.

This reads **real samples from the file**, not the live buffer, because the
live buffer is an envelope and its intra-block order is not real.

* `RunFiles(outdir, channel)` — the run's files as they grow. Re-reads the
  manifest and file sizes on demand, because a cached size is a lie 200 ms
  later, and finds the **open** segment by name since it has no manifest entry
  until it closes.
* `files.n_scans()` — how much has been written so far.
* `files.read(start_scan, n)` — just that window, one seek and one read, across
  segment joins. A 10 s window of a week-long run touches about a megabyte.

In [ ]:
files = RunFiles(OUTDIR, "ai0")
rate = files.rate

SPAN, PRE = 1500, 100        # samples on screen, samples shown before the trigger

def trigger(y, level=0.0, pre=PRE, n=SPAN):
    """Align on the first rising crossing - a scope trigger, four lines."""
    above = y >= level
    hits = np.flatnonzero(~above[:-1] & above[1:]) + 1
    hits = hits[hits >= pre]                 # need room to show the run-up
    if len(hits) == 0:
        return y[:n]                         # free-run if nothing crosses
    return y[hits[0] - pre : hits[0] - pre + n]

fig, ax = plt.subplots(figsize=(12, 4))
line, = ax.plot([], [], lw=1)
ax.set_xlabel("ms"); ax.set_ylabel("V"); ax.grid(alpha=0.3)

try:
    while run.alive:
        total = files.n_scans()
        y = files.read(max(0, total - 5000), 5000)     # the last 0.2 s
        if len(y) < SPAN:
            time.sleep(0.2); continue
        y = trigger(y)
        line.set_data(np.arange(len(y)) / rate * 1e3, y)
        ax.set_xlim(0, len(y) / rate * 1e3)
        ax.set_ylim(y.min() - 0.1, y.max() + 0.1)
        ax.set_title(f"{total / rate:,.0f} s recorded")
        clear_output(wait=True); display(fig)
        time.sleep(0.2)
except KeyboardInterrupt:
    print("stopped")

## 5.4 `LiveScope` — navigate the whole run while it records

```python
scope = LiveScope(outdir, channel="ai0", width=5.0)
scope.show()          # follows the end of the file; interrupt to stop
scope.goto(120.0)     # jump to t = 120 s, stop following
scope.zoom(0.1)       # 10x narrower
scope.pan(-1)         # one window back
scope.follow()        # back to live
scope.step()          # draw ONE frame, for your own loop
```

It draws **min/max per pixel column**, which is what a digital scope does and
for the same reason: a 300 s window is 7.5 M samples against ~1500 pixels, and
taking every 5000th sample would drop a 200 µs spike about 999 times in 1000.
Measured on a synthetic run with fifteen 200 µs spikes: min/max keeps 15 of 15,
striding keeps **0**. Zoom in past one sample per column and it stops
decimating entirely.

Nothing in `livescope` touches the DAQ, so a slow plot cannot stall the
acquisition — you can even run it in another process.

In [ ]:
scope = LiveScope(OUTDIR, channel="ai0", width=0.05)   # 50 ms: no decimation
scope.show()          # interrupt the kernel to stop

In [ ]:
run.stop()            # closes the current file cleanly
daq.ao.stop()         # waveform off, ao0 back to 0 V
print("alive:", run.alive)

---
# 5. Markers — recording *when* things happened

The 1 pps gives you a **ruler**: every edge is one atomic second, and you know
the scan index of each. What it does not tell you is where the interesting
moments are — when the OPX started a run, stopped one, or paused to drain its
buffer.

`daq.marker` is the other half. It is the **same counter trick** as the 1 pps,
on the free counter, pointed at a different wire:

| wire | counter | what you get |
|---|---|---|
| rubidium 1 pps | `ctr1` (`daq.pps`) | atomic seconds — the ruler |
| OPX marker line | `ctr0` (`daq.marker`) | events — when things happened |

Both record **scan indices**, and that is the whole point: markers, 1 pps
edges, samples and segment boundaries are all the same kind of number. Relating
an event to the data is `y[mark]`; relating it to the rubidium is one call.
Nothing is converted, so nothing can be converted wrong.

**Why the scan index and not a time.** A time would have to be derived by
dividing by a rate that is itself wrong by ~12 ppm, and frozen into the file at
whatever the driver's clock correction happened to be that day. A scan index is
what the hardware measures, it is an exact integer, it indexes the data
directly, and it can be **re-timed** when the correction improves — which it
has, twice.

**One line, one kind of pulse.** Only one counter is free, so the hardware
cannot tell "start" from "stop". Let **order** carry the meaning: your OPX
program defines the sequence, so mark 0 is whatever it emits first. The DAQ
only has to say *when*.

**What it buys.** The DAQ records continuously; the OPX does not. The markers
say which stretches of an unbroken recording correspond to the OPX doing
something — and which are the gaps while it reset or drained. The DAQ becomes
a witness for an instrument that stops, because it never does.

## 5.1 A bench trial — the board triggers and marks itself

Before wiring the OPX, simulate it: use two analog outputs as digital lines.
This exercises three things that have never been tested together — **markers**,
**two AI channels at once**, and **a trigger arriving while the counter waits**.

### Wiring

| from | to | why |
|---|---|---|
| **AO 0** (15) | **PFI 8** (81) + **AI 1** (4) | the "QM trigger" — and AI 1 watches it |
| **AO 1** (31) | **PFI 10** (85) + **AI 0** (1) | the "QM markers" — and AI 0 watches them |
| rubidium 1 pps | **PFI 9** (83) | the ruler |

Grounds: AO GND 16 → AI GND 3, and D GND 82 / 84 for the PFI returns.

Wiring each output to a PFI line **and** an AI channel is the trick that makes
this a real test: the marker counter records a scan index, and AI 0 records the
pulse itself. If the two agree, the whole chain is verified.

### ⚠ Never drive a PFI input negative

PFI lines are **TTL inputs**, absolute maximum about **−0.5 V to +5.5 V**. The
±1 V square used earlier in this notebook would put −1 V on the pin, which is
outside that. For this trial the outputs only ever sit at **0 V or 5 V**.

### Why DC levels and not a waveform

Setting a DC level while a waveform is running rebuilds the AO task, which
would restart and glitch the waveform. In **pure DC mode** — no channel with
role `"wave"` — each `dc()` is an independent untimed write with nothing to
restart, so you can pulse a line from Python whenever you like. That is also
much closer to what the OPX will actually do.

In [ ]:
# two INPUTS: ai0 watches the marker line, ai1 watches the trigger line.
# 0..5 V signals, so give them room; DAQmx snaps up to the nearest range.
ai0, ai1 = daq.ai.setup({"ai0": (-1.0, 6.0), "ai1": (-1.0, 6.0)},
                        rate=25_000, trigger="/Dev1/PFI8")

# two OUTPUTS as digital lines. Pure DC mode: NO channel has role "wave",
# so each dc() is an untimed write and nothing gets restarted.
ao0, ao1 = daq.ao.setup(channels_role={"ao0": {"dc": 0.0},    # trigger, idle low
                                       "ao1": {"dc": 0.0}})   # markers, idle low

daq.pps.terminal("/Dev1/PFI9")       # the ruler
daq.marker.terminal("/Dev1/PFI10")   # the events
daq.pps.counter("ctr1"); daq.marker.counter("ctr0")

daq.check(strict=True)
daq.terminal_roles()

In [ ]:
daq.wiring()

# and the two channels are not sampled at the same instant - one ADC, one mux
for ch in daq.ai.active:
    print(f"{ch.short_name}: sampled {ch.time_offset()*1e6:5.2f} us into each scan")

### Run it

The order matters and mirrors the real thing:

1. both lines **low**, so the first edge is a clean rising one
2. `long_run` starts — the counter begins immediately, but the **AI task waits**
   for PFI 8, so every 1 pps edge during the wait latches 0 and is discarded
3. we raise AO 0 → that is the trigger, and **scan 0 is that instant**
4. we pulse AO 1 a few times → those are the markers

In [ ]:
import time

OUTDIR = Path("marker_trial")          # a NEW directory

ao0.dc(0.0); ao1.dc(0.0)               # both idle low
time.sleep(0.2)

run = daq.long_run(OUTDIR, hours=40/3600,      # 40 seconds
                   background=True, verbose=False)
time.sleep(3)                           # let the AI task arm and WAIT
print("armed; the counter is running but the AI clock is not")

ao0.dc(5.0)                             # <-- TRIGGER: rising edge on PFI 8
print("triggered - scan 0 is now")

for i in range(4):                      # four markers, 6 s apart
    time.sleep(6)
    ao1.dc(5.0); time.sleep(0.02); ao1.dc(0.0)     # a 20 ms pulse
    print(f"   marker {i} sent")

run.join()
ao0.dc(0.0)
print("done")

### Read it back — and check the markers against the signal

`marker_times` gives the scan index and atomic second of every marker. AI 0
recorded the same pulses as a waveform, so finding its rising edges gives an
*independent* answer. **They should agree to about one scan (40 µs).**

In [ ]:
manifest, edges = load_long_run(OUTDIR)
rate = manifest[0]["nominal_rate"]

print("=== timing ===")
edge_report(edges, nominal_rate=rate)
info = trigger_time(OUTDIR)
print(f"\ntrigger fired {info['first_edge_seconds']:.4f} s before the first "
      f"atomic second")

print("\n=== markers ===")
marks, t_marks = marker_times(OUTDIR)
for i, (m, t) in enumerate(zip(marks, t_marks)):
    print(f"   marker {i}: scan {m:>9,}   atomic {t:+9.4f} s")
print(f"   spacing: {np.diff(t_marks).round(3)} s   (6 s apart by construction)")

In [ ]:
# INDEPENDENT CHECK: ai0 watched the marker line, so find its rising edges
idx, y = load_run(OUTDIR, "ai0")
above = y >= 2.5
seen = np.flatnonzero(~above[:-1] & above[1:]) + 1 + idx[0]   # global scans

print(f"counter recorded : {marks.tolist()}")
print(f"seen in ai0      : {seen.tolist()}")

if len(seen) == len(marks):
    delta = marks - seen
    print(f"difference       : {delta.tolist()} scans "
          f"= {(delta / rate * 1e6).round(1).tolist()} us")
    print("\nAgreement within a scan or two means the whole chain is right:")
    print("  the counter latched the same instant the ADC saw the edge.")
else:
    print(f"\nMISMATCH: {len(marks)} marks vs {len(seen)} edges in the data.")
    print("  more edges than marks -> the pulse bounced, or ai0 sees noise")
    print("  fewer -> the pulse missed the PFI threshold; check the wiring")

In [ ]:
# and the picture: the marker pulses with their recorded positions on top
t_atomic = times_from_edges(idx, edges, scans_per_second=rate)

fig, ax = plt.subplots(figsize=(13, 4))
step = max(1, len(y) // 200_000)
ax.plot(t_atomic[::step], y[::step], lw=0.8, label="ai0 (marker line)")
for i, t in enumerate(t_marks):
    ax.axvline(t, color="red", ls="--", lw=1.5,
               label="counter marks" if i == 0 else None)
ax.set_xlabel("atomic time [s]"); ax.set_ylabel("V"); ax.legend()
ax.grid(alpha=0.3)
ax.set_title("red lines = what the counter recorded, trace = what the ADC saw")

### What each outcome tells you

| result | meaning |
|---|---|
| marks and edges agree within a scan or two | the whole chain works — wire the OPX in |
| more edges than marks | the pulse bounced, or AI 0 is picking up noise: raise the 2.5 V threshold |
| fewer edges than marks | the pulse did not reach the PFI threshold — check the D GND return |
| no marks at all | `daq.marker.terminal` unset, or the counter never claimed — `daq.terminal_roles()` |
| `t_marks` starts negative | normal: the markers came before the first atomic second |

Once this passes, replacing AO 0 and AO 1 with the OPX's trigger and marker
outputs changes **nothing** in the analysis — same scan indices, same edge
table, same two lines of read-back.

---
# 6. Reading a run back

Everything here is in `postprocess`, which needs **no hardware and no
NI-DAQmx** — you can do this on any machine.

### `load_long_run(outdir)` → `(manifest, edges)`

* `manifest` — the parsed `manifest.jsonl`: a header row (`event="start"` with
  the channels and nominal rate), one row per **file**, and one row per **gap**
  where an error interrupted the run.
* `edges` — **every** 1 pps edge of the whole run, raw. One file for the run,
  not per segment and not per channel, because edges are global scan indices
  and the sample clock ticks once per scan whatever the channel count.

`gaps` must be 0. A gap means the acquisition restarted: the samples either
side are adjacent in scan index but **not** contiguous in time, and only the
manifest records that.

In [ ]:
manifest, edges = load_long_run(OUTDIR)
rate  = manifest[0]["nominal_rate"]
gaps  = [e for e in manifest if e.get("event") == "gap"]

print("channels:", channels_of(OUTDIR))
print("segments:", sorted({e["index"] for e in manifest if "file" in e}))
print("gaps    :", len(gaps), " <- must be 0")
for g in gaps:
    print(f"   at scan {g['sample']:,}: {g['error']}")

print()
report = edge_report(edges, nominal_rate=rate)

### Getting the samples

| function | returns |
|---|---|
| `load_segment(outdir, index, channel, mmap=False)` | **one** segment. `index` is the segment number — 0 is the first file, 1 the second |
| `iter_segments(outdir, channel)` | yields `(scans, samples)` per segment — walk a long run without holding it all |
| `load_run(outdir, channel)` | the whole channel concatenated. 360 MB per hour at 25 kS/s — check first |

All three return **global scan indices** alongside the samples, so the pieces
line up end to end. `mmap=True` leaves the samples on disk and pages in only
what you touch, so a multi-gigabyte segment plots without the RAM.

### `times_from_edges(scans, edges, scans_per_second=None, clean=True)`

Turns scan indices into seconds since the first atomic second. It interpolates
between the **surrounding edges** rather than dividing by a rate, which is what
absorbs the drift.

Two rules:

* **Pass the whole edge table**, never a slice. The numbering starts from the
  first edge it is given, so slicing to "the edges inside this segment" would
  throw away the seconds that came before and shift everything.
* **It cleans by default.** The raw table contains the counter's start value
  and the end-of-acquisition value; leaving them in shifted a 300 s record to
  1.000 … 301.427 s. `clean=False` opts out and raises if the table needs it.

In [ ]:
# mmap=True keeps a multi-gigabyte segment on disk instead of in RAM
idx, y = load_segment(OUTDIR, 0, "ai0", mmap=True)

t_atomic = times_from_edges(idx, edges, scans_per_second=rate)  # cleans by default
t_board  = idx / rate

print(f"{len(y):,} scans in segment 0")
print(f"atomic {t_atomic[0]:+.4f} .. {t_atomic[-1]:,.4f} s")
print(f"board  {t_board[0]:+.4f} .. {t_board[-1]:,.4f} s")

In [ ]:
# walk every segment without loading the whole run
for idx_s, y_s in iter_segments(OUTDIR, "ai0"):
    print(f"segment: scans {idx_s[0]:>12,} .. {idx_s[-1]:>12,}  "
          f"{len(y_s):>10,} samples  mean {y_s.mean(dtype=np.float64):+.4f} V")

In [ ]:
start, end = -10000, -1

plt.figure(figsize=(12, 4))
plt.plot(t_atomic[start:end], y[start:end], lw=0.8)
plt.xlabel("atomic time [s]"); plt.ylabel("V [V]"); plt.grid(alpha=0.3)

### The drift curve — what the run was really for

`np.diff(kept)` is scans per atomic second, second by second. That *is* the
board's clock being measured against an atomic reference, and it is the plot
worth keeping from a long run.

In [ ]:
kept, dropped = clean_edges(edges, rate)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(len(kept) - 1), np.diff(kept), lw=1)
ax.axhline(rate, color="k", ls="--", lw=1, label=f"nominal {rate:,.0f}")
ax.set_xlabel("atomic second"); ax.set_ylabel("scans / atomic second")
ax.legend(); ax.grid(alpha=0.3)

print(f"{len(dropped)} artefact(s) discarded: {dropped}")
print(f"clock error {report['ppm']:+.2f} ppm -> {report['seconds_per_day']:.2f} s/day")

---
# 7. Reference

### Configuring

| call | does |
|---|---|
| `daq.ai.setup(chans, rate=, duration=, trigger=, terminal_config=)` | configure the input, return the channels |
| `daq.ao.setup(channels_role=, shape=, freq=, amp=, offset=, rate=, duty=, trigger=)` | configure the output, return the channels |
| `daq.pps.terminal(...)`, `daq.pps.counter(...)` | the 1 pps line and which counter latches it |
| `daq.check(strict=False)` | validate everything without touching hardware |
| `daq.wiring()` | the screw-terminal table, printed and returned |

### Measuring

| call | does |
|---|---|
| `daq.ai.acquire()` | one record, `{channel: array}`. The only thing that measures |
| `daq.ai.acquire_chunks(duration=None)` | stream it instead, chunk by chunk, constant memory |
| `daq.acquire_with_pps()` | one record **plus** the atomic seconds inside it |
| `daq.clock_check(seconds=)` | measure the board's clock against the rubidium |
| `ai0.trace()`, `daq.ai.time_axis()`, `daq.ai.atom_time_axis()` | the QCoDeS parameter view of the last record |

### Generating

| call | does |
|---|---|
| `ao0.dc(v)` | a constant voltage, applied immediately |
| `daq.ao.start()` / `daq.ao.stop()` | start / stop the waveform |
| `with daq.ao.generating(**overrides):` | generate for a block, stop guaranteed |
| `daq.ao.onboard()` | True if the waveform loops from board memory |

### Recording and reading

| call | does |
|---|---|
| `daq.long_run(outdir, hours=, rotate_minutes=, background=)` | record to disk |
| `run.status()`, `run.alive`, `run.stop()` | the background handle |
| `daq.live_preview()` | the last 60 s of one channel, from RAM |
| `LiveScope(outdir, channel, width)` | navigate the files while they grow |
| `load_long_run`, `load_segment`, `iter_segments`, `load_run` | read it back |
| `edge_report`, `clean_edges`, `times_from_edges` | the timing |
| `daq.describe()`, `daq.snapshot()` | what everything was set to |

# 8. When something goes wrong

| symptom | cause |
|---|---|
| an edit to the driver changes nothing | **restart the kernel** — Python caches imported modules |
| ppm is enormous, `t_atomic` starts at 1.0 | same: a stale kernel, so the edge cleaning is not running |
| the live plot never updates | missing `clear_output`/`display`, or `plt.pause` on the inline backend |
| the live waveform looks ragged | you are looking at a decimated summary — use `LiveScope` or §5.3 |
| `gaps` is not 0 | usually `-200279` (the PC did not read fast enough) or `-200361` (USB could not drain the board) |
| AO underflow `-200621` | `daq.ao.onboard()` was False — lower `ao.rate` or raise `ao.freq` |
| few or no 1 pps edges | check `daq.wiring()` — is the pulse on the terminal you think? |
| `FileExistsError` on `long_run` | that directory already holds a run. Use a new one |
| `KeyError: no segment N` | the message lists the segments that do exist |
| overflow after adding a channel | the rate limit is **aggregate** — `daq.check()` does the arithmetic |

**Keep the raw output directory.** Every number in this notebook can be
recomputed from it.

In [ ]:
daq.close()